In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# 1. Charger le dataset
df = pd.read_csv("Public_Chauffeurs_20260405.csv")

# 2. Colonnes utiles
df = df[['Renewed', 'Status', 'Driver Type', 'License Type', 'Sex', 'Chauffeur City']]

# 3. Encodage
for col in df.columns:
    df[col] = LabelEncoder().fit_transform(df[col].astype(str))

# 4. Normalisation
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df)

# 5. K-Means
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
labels = kmeans.fit_predict(X_scaled)

# 6. SSE (inertia)
SSE = kmeans.inertia_
print("SSE (Intra-cluster) :", SSE)

# 7. SSB (Between-cluster)
overall_mean = np.mean(X_scaled, axis=0)

SSB = 0
for i in range(3):
    cluster_points = X_scaled[labels == i]
    cluster_mean = np.mean(cluster_points, axis=0)
    ni = len(cluster_points)
    SSB += ni * np.sum((cluster_mean - overall_mean) ** 2)

print("SSB (Inter-cluster) :", SSB)

# 8. Ratio (optionnel)
print("Ratio SSB/SSE :", SSB / SSE)

# 9. PCA pour visualisation
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

# 10. Visualisation
plt.figure(figsize=(8,6))
plt.scatter(X_pca[:,0], X_pca[:,1], c=labels, cmap='viridis', s=50)

plt.title("K-Means Clustering - Chauffeurs Dataset")
plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.show()

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# 1. Charger le dataset
df = pd.read_csv("Public_Chauffeurs_20260405.csv")

# 2. Sélection des colonnes utiles
df = df[['Renewed', 'Status', 'Driver Type', 'License Type', 'Sex', 'Chauffeur City']]

# 3. Encodage
for col in df.columns:
    df[col] = LabelEncoder().fit_transform(df[col].astype(str))

# 4. Normalisation
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df)

# 5. K-Means++
kmeans = KMeans(n_clusters=3, init='k-means++', random_state=42, n_init=10)
labels = kmeans.fit_predict(X_scaled)

# 6. SSE (intra-cluster)
SSE = kmeans.inertia_
print("SSE :", SSE)

# 7. SSB (inter-cluster)
overall_mean = np.mean(X_scaled, axis=0)

SSB = 0
for i in range(3):
    cluster_points = X_scaled[labels == i]
    cluster_mean = np.mean(cluster_points, axis=0)
    ni = len(cluster_points)
    SSB += ni * np.sum((cluster_mean - overall_mean) ** 2)

print("SSB :", SSB)

# 8. Ratio
print("Ratio SSB/SSE :", SSB / SSE)

# 9. PCA pour visualisation
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

# 10. Visualisation
plt.figure(figsize=(8,6))
plt.scatter(X_pca[:,0], X_pca[:,1], c=labels, cmap='plasma', s=50)

plt.title("K-Means++ Clustering - Chauffeurs Dataset")
plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.show()

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import DBSCAN
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# Charger le dataset
df = pd.read_csv("Public_Chauffeurs_20260405.csv")

# Colonnes utiles
df = df[['Renewed', 'Status', 'Driver Type', 'License Type', 'Sex', 'Chauffeur City']]

# Encodage
for col in df.columns:
    df[col] = LabelEncoder().fit_transform(df[col].astype(str))

# Normalisation
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df)

# DBSCAN
dbscan = DBSCAN(eps=1.5, min_samples=5)
labels = dbscan.fit_predict(X_scaled)

#  Ignorer le bruit (-1)
mask = labels != -1
X_clean = X_scaled[mask]
labels_clean = labels[mask]

# Calcul SSE
SSE = 0
unique_clusters = set(labels_clean)

for cluster in unique_clusters:
    points = X_clean[labels_clean == cluster]
    centroid = np.mean(points, axis=0)
    SSE += np.sum((points - centroid) ** 2)

print("SSE :", SSE)

# Calcul SSB
overall_mean = np.mean(X_clean, axis=0)
SSB = 0

for cluster in unique_clusters:
    points = X_clean[labels_clean == cluster]
    centroid = np.mean(points, axis=0)
    ni = len(points)
    SSB += ni * np.sum((centroid - overall_mean) ** 2)

print("SSB :", SSB)

# Ratio
if SSE != 0:
    print("Ratio SSB/SSE :", SSB / SSE)

# PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

# Visualisation
plt.figure(figsize=(8,6))
plt.scatter(X_pca[:,0], X_pca[:,1], c=labels, cmap='viridis', s=50)

plt.title("DBSCAN Clustering - Chauffeurs Dataset")
plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.show()

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import AgglomerativeClustering
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import dendrogram, linkage

# 1. Charger le dataset
df = pd.read_csv("Public_Chauffeurs_20260405.csv")

# 2. Colonnes utiles
df = df[['Renewed', 'Status', 'Driver Type', 'License Type', 'Sex', 'Chauffeur City']]

# 3. Encodage
for col in df.columns:
    df[col] = LabelEncoder().fit_transform(df[col].astype(str))

# 4. Échantillon
df = df.sample(2000, random_state=42)

# 5. Normalisation
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df)

# 6. Dendrogramme
linked = linkage(X_scaled, method='ward')

plt.figure(figsize=(10,6))
dendrogram(linked)
plt.title("Dendrogramme - Clustering Hiérarchique")
plt.show()

# 7. Clustering
hc = AgglomerativeClustering(n_clusters=3, linkage='ward')
labels = hc.fit_predict(X_scaled)

# 8. SSE (intra-cluster)
SSE = 0
for cluster in set(labels):
    points = X_scaled[labels == cluster]
    centroid = np.mean(points, axis=0)
    SSE += np.sum((points - centroid) ** 2)

print("SSE :", SSE)

# 9. SSB (inter-cluster)
overall_mean = np.mean(X_scaled, axis=0)
SSB = 0

for cluster in set(labels):
    points = X_scaled[labels == cluster]
    centroid = np.mean(points, axis=0)
    ni = len(points)
    SSB += ni * np.sum((centroid - overall_mean) ** 2)

print("SSB :", SSB)

# 10. Ratio
print("Ratio SSB/SSE :", SSB / SSE)

# 11. PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

# 12. Visualisation
plt.figure(figsize=(8,6))
plt.scatter(X_pca[:,0], X_pca[:,1], c=labels, cmap='tab10', s=50)

plt.title("Clustering Hiérarchique - Chauffeurs Dataset")
plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.show()